# Genetic Algorithm for 2x2 Jigsaw Puzzle

This notebook demonstrates a simple Genetic Algorithm (GA) to solve a synthetic 2x2 jigsaw puzzle. 

## Problem Description
- We have 4 puzzle pieces arranged as a 2x2 grid.
- Each piece has integer values representing edges.
- Goal: find an arrangement where adjacent edges match as closely as possible.

We define a **fitness function** to evaluate how well the pieces fit together.

## Fitness Function
Let $p_i$ be the piece at position $i$. The edge fit is defined as:

$$\text{fit}(p_i, p_j, \text{side}) = \sum | \text{edge of } p_i - \text{corresponding edge of } p_j |$$

The total puzzle fitness is:

$$\\text{fitness} = \\text{max\_diff} - \\sum \\text{fit along all adjacent edges}$$
where higher fitness means better assembly.

In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt

# --- Puzzle piece abstraction ---
class Piece:
    def __init__(self, id, data):
        self.id = id
        self.data = data

# --- Synthetic 2x2 puzzle pieces ---
pieces = [
    Piece(0, np.array([[1,2],[3,4]])),
    Piece(1, np.array([[2,5],[4,6]])),
    Piece(2, np.array([[5,7],[6,8]])),
    Piece(3, np.array([[7,9],[8,10]]))
]
N = len(pieces)

# --- Fitness function ---
def edge_fit(piece1, piece2, side):
    if side == 'right':
        diff = np.abs(piece1.data[:, -1] - piece2.data[:, 0])
    elif side == 'bottom':
        diff = np.abs(piece1.data[-1, :] - piece2.data[0, :])
    return np.sum(diff)

def puzzle_fitness(arrangement):
    max_diff = 100
    total_diff = edge_fit(arrangement[0], arrangement[1], 'right')
    total_diff += edge_fit(arrangement[0], arrangement[2], 'bottom')
    total_diff += edge_fit(arrangement[1], arrangement[3], 'bottom')
    total_diff += edge_fit(arrangement[2], arrangement[3], 'right')
    return max_diff - total_diff

ModuleNotFoundError: No module named 'numpy'

## GA Helper Functions
- **Selection:** tournament selection.
- **Crossover:** single-point sequence crossover.
- **Mutation:** swap pieces randomly with probability 0.5 to maintain diversity.

These operators implement the **evolutionary loop** of the GA.

In [1]:
def array_in_list(piece, lst):
    return any(piece.id == x.id for x in lst)

def crossover(parent1, parent2):
    point = random.randint(1, N-1)
    child = parent1[:point] + [p for p in parent2 if not array_in_list(p, parent1[:point])]
    return child

def mutate(arrangement, mutation_rate=0.5):
    arr = arrangement.copy()
    for i in range(N):
        if random.random() < mutation_rate:
            j = random.randint(0, N-1)
            arr[i], arr[j] = arr[j], arr[i]
    return arr

def selection(population, fitnesses, num_selected):
    selected = []
    for _ in range(num_selected):
        i, j = random.sample(range(len(population)), 2)
        selected.append(population[i] if fitnesses[i] > fitnesses[j] else population[j])
    return selected

## GA Parameters and Initialization
- Population size: 10
- Generations: 20
- Initial population: random arrangements of the 4 pieces

In [ ]:
pop_size = 10
generations = 20
population = [random.sample(pieces, N) for _ in range(pop_size)]
best_fitness_history = []

## Main GA Loop##
At each generation:
1. Compute fitness of all individuals.
2. Print number of unique arrangements (to track diversity).
3. Select parents using tournament selection.
4. Apply crossover and mutation to create next generation.
5. Track best fitness over generations.

In [ ]:
for gen in range(generations):
    fitnesses = [puzzle_fitness(ind) for ind in population]
    best_idx = np.argmax(fitnesses)
    best_fitness_history.append(fitnesses[best_idx])
    unique_ids = len(set(tuple(p.id for p in ind) for ind in population))
    print(f"Generation {gen}: unique arrangements = {unique_ids}")
    print(f"Generation {gen}: best fitness = {fitnesses[best_idx]}, arrangement = {[p.id for p in population[best_idx]]}")

    selected = selection(population, fitnesses, pop_size)
    next_pop = []
    for i in range(0, pop_size, 2):
        p1, p2 = selected[i], selected[(i+1)%pop_size]
        next_pop.extend([mutate(crossover(p1, p2)), mutate(crossover(p2, p1))])
    population = next_pop[:pop_size]

## Results
After running 20 generations, we select the best puzzle arrangement and plot the fitness evolution over generations.

In [ ]:
fitnesses = [puzzle_fitness(ind) for ind in population]
best_idx = np.argmax(fitnesses)
best_solution = population[best_idx]

print("\nBest puzzle arrangement (piece IDs):")
print([p.id for p in best_solution])

# Plot fitness evolution
plt.plot(best_fitness_history, marker='o')
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('GA Fitness Evolution')
plt.grid(True)
plt.show()

As seen from above, 
Best puzzle arrangement (piece IDs):
[0, 1, 2, 3]

Best arrangement is found to be [0,1,2,3]
We can also see the plot visualizing the result of each arrangement.